# RiskMatch research replays

Local development experiments preserved in execution order. Install the pilot requirements and provide the authorized data and course files before running from the repository root. Dataset scopes, dates and allocation names apply to the individual experiment. See [the implementation checklist](CHECKLIST.md) for current product decisions.

### Step 0 — Setup

In [ ]:
from pathlib import Path
import csv
import yaml

setup = yaml.safe_load(Path("config.yaml").read_text())
with Path("experiments.csv").open(newline="") as source:
    studies = list(csv.DictReader(source))
expected = {f"H{i:02d}" for i in range(1, 8)} | {"ETF-BENCH-v1"}
assert {row["id"] for row in studies} == expected and len(studies) == len(expected)
print("Setup version", setup["version"], "·", len(studies), "registered study designs")

### 0.1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import cvxpy as cp
from scipy.stats import mannwhitneyu
from sklearn.covariance import LedoitWolf
from pathlib import Path

### 0.2 — Configuration (tickers, dates, SEED, assumptions)

In [ ]:
benchmark = setup["benchmark"]
benchmark_assets = benchmark["assets"]
pilot_start, pilot_end = map(pd.Timestamp, benchmark["development"])
prospective_start, prospective_end = map(pd.Timestamp, benchmark["prospective"])
cov_window = benchmark["covariance_sessions"]
print("ETF development and prospective windows:", benchmark["development"], benchmark["prospective"])

### 0.3 — Folder Paths (data/, results/, questions/)

In [ ]:
for directory in ("riskmatch", "questions", "tests", "data/demo", "research"):
    assert Path(directory).is_dir(), directory
print("The course exercises can be reproduced separately with python -m research.course_labs.")

### 1.1 — Price inventory by study universe and benchmark

In [ ]:
# The archived ETF panel supports the benchmark replay. Register additional instruments
# per hypothesis with their identifier, coverage, currency, tradability and point-in-time constraints.

### Control – Part 01 — price and data integrity

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import math
from datetime import date

# Run from the root of the project.
data_dir = Path("data")
with (data_dir / "raw/etf_prices_adjusted.csv").open() as stream:
    reader = csv.DictReader(stream)
    assets = [name for name in reader.fieldnames if name != "date"]
    rows = list(reader)
dates = [date.fromisoformat(row["date"]) for row in rows]
assert dates and dates == sorted(set(dates)), "Duplicate or unordered dates"
assert all(math.isfinite(float(row[a])) and float(row[a]) > 0
           for row in rows for a in assets), "Missing, non-finite or nonpositive price"
manifest = json.loads((data_dir / "processed/common_sample_manifest.json").read_text())
for filename, entry in manifest["files"].items():
    path = data_dir / "processed" / filename
    assert hashlib.sha256(path.read_bytes()).hexdigest() == entry["sha256"], filename
print(f"{len(dates)} sessions, {len(assets)} series, {dates[0]} to {dates[-1]}")
print(f"{len(manifest['files'])} checksums verified. Final test starts: {manifest['final_test_start']}")
from data_audit import audit_local, verify_calendar
audit_local()


### 3.0b — Separate goals, willingness, financial capacity, hard exclusions, preferences and follow-up

In [ ]:
from riskmatch.intake import sample, assess
from riskmatch.profile_session import Interview

# One fictional CHF mandate. No market data or client records are needed for Step 3.
profile_facts = sample()
profile_facts.update(goal="70000", contribution="250", wish="3")
demo_interview = Interview(profile_facts, seed=setup["seeds"]["profile_example"], pilot=True)
print("Mandate:", demo_interview.facts["project"], "·", demo_interview.facts["currency"])

### 3.1d — Version the bank and validate 60 unique cards, 15 common and 45 conditional

In [ ]:
from pathlib import Path
from collections import Counter
import json
import math

bank = json.loads(Path("questions/bank.json").read_text())
transfer = json.loads(Path("questions/transfer_validation.json").read_text())
cards = bank["cards"]
card_ids = [card["id"] for card in cards]
assert bank["language"] == "en" and len(set(card_ids)) == len(cards) == 60
assert sum(card["core"] for card in cards) == 15
assert Counter(key[0] for key in card_ids) == dict(C=2, A=20, L=14, B=10, F=8, S=6)
assert len(transfer["items"]) == len({item["id"] for item in transfer["items"]}) == 4
assert set(card_ids).isdisjoint(item["id"] for item in transfer["items"])
for card in cards:
    if card["id"].startswith("A"):
        outcomes = [amount / card["stake"] - 1 for amount in card["outcomes"]]
        probabilities = card["probabilities"]
        assert math.isclose(sum(probabilities), 1)
        mean = sum(r * p for r, p in zip(outcomes, probabilities, strict=True))
        variance = sum(p * (r - mean)**2 for r, p in zip(outcomes, probabilities))
        certain_return = card["certain"] / card["stake"] - 1
        threshold = 2 * (mean - certain_return) / variance
        assert math.isclose(threshold, card["phi_threshold"], abs_tol=1e-10)
print(f'{bank["version"]}: 60 English cards, 15 core, 45 conditional. '
      'Verified 20 financial thresholds and 4 separate transfer items.')


### 3.2 — Estimate a range for φ from understood financial choices only; handle indifference, skipped and misunderstood choices

In [ ]:
from riskmatch.profile import summary

financial_items = demo_interview.cards
for understood_choices in ({"A01": 0, "A02": 1}, {"A01": 1, "A04": 0}, {"A01": -1, "A02": 3}):
    estimate = summary(financial_items, understood_choices)
    print(understood_choices, estimate["reason"] or (estimate["lo"], estimate["hi"], estimate["bands"]))

### 3.4 — Encode answer-specific triggers across families, common coverage and correction of earlier answers

In [ ]:
# Simulate a coherent preference ONLY to exercise the route. The value is known by construction.
while demo_interview.current()[0] == "profile":
    _, item_id = demo_interview.current()
    card = demo_interview.items[item_id]
    if item_id.startswith("C"):
        choice = card["answers"].index(card["correct"])
    elif item_id.startswith("A"):
        choice = int(3 < card["phi_threshold"])
    else:
        choice = int(item_id == "F01")
    statement = demo_interview.statement()
    direction = "right" if statement["right"] == choice else "left"
    demo_interview.apply(dict(action="swipe", id=item_id, step=demo_interview.step,
                              direction=direction, seconds=0))
assert demo_interview.current()[0] == "validation"
assert all(trial["id"] not in demo_interview.answers for trial in demo_interview.frozen)
print(len(demo_interview.answers), "profile cards before the four independent choices")
print(" → ".join(demo_interview.answers))
print("Frozen before validation:", demo_interview.frozen)

### 3.6 — Require two distinct comparisons and client confirmation for each reported behavioral observation (maximum two)

In [ ]:
loss_comparisons = dict(L01=1, L02=0, L03=1, L04=0)
assert summary(financial_items, loss_comparisons)["observation"] is None
print(summary(financial_items, dict(loss_comparisons, B09=0))["observation"])

### 3.7 — Assess project capacity from dated withdrawals, reserves and obligations; apply hard constraints separately

In [ ]:
for variant in (profile_facts, dict(profile_facts, withdraw="48000", due="0",
                household="Reserve needed from this investment", safety="5000", exclusions="Tobacco")):
    capacity_record = assess(variant, {})
    print("Protected / investable / shortfall in CHF:",
          capacity_record["reserve"], capacity_record["free"], capacity_record["funding_gap"])
    print("Required annual growth rate:", capacity_record["required_rate"])
    print("Unresolved constraints:", capacity_record["reasons"])

### 3.9 — Test routing, termination, scoring independence, incomplete profiles, export/import and revised answers

In [ ]:
# Validation choices do not update tolerance. An imported score is reconstructed from events.
frozen_tolerance = demo_interview.report()["tolerance"]
for trial in demo_interview.frozen:
    while demo_interview.current()[1] == trial["id"]:
        demo_interview.apply(dict(action="swipe", id=trial["id"], step=demo_interview.step,
                                  direction="pass", seconds=0))
demo_interview.apply(dict(action="confirm"))
restored_interview = Interview.restore(demo_interview.dump())
assert restored_interview.report() == demo_interview.report()
assert restored_interview.report()["tolerance"] == frozen_tolerance
restored_interview.apply(dict(action="revise", id="B01"))
assert not restored_interview.confirmed and "B01" not in restored_interview.answers
print("Verified confirmation, JSON round trip, correction and frozen tolerance.")

### 3.11 — Record calibration and limitations; compare common-only vs adaptive results before claiming validity

In [ ]:
from riskmatch.pilot import evaluate

# No human participants have been enrolled in this notebook. Empty denominators remain undefined.
empty_pilot = evaluate([])
assert empty_pilot["predictions"]["transfer"]["scores"]["adaptive"]["agreement"] is None
print("Human calibration pending. No timing, agreement or retest estimate is available.")
print("Synthetic example — core bands:", demo_interview.report()["core_tolerance"]["bands"],
      "adaptive bands:", frozen_tolerance["bands"])

### H05 — Does WRDS stock selection add value after costs?

In [ ]:
from data_audit import h05_panel_at_close

wrds_path = Path("data/additional_data/clean/monthly_panel.parquet")
if wrds_path.exists():
    h05_panel = h05_panel_at_close(wrds_path)
    print("Book-to-market observable under the assumed lag:", h05_panel.bm.notna().sum(), "/", len(h05_panel))
else:
    print("Private panel unavailable. H05 pending.")

### 4.1 — Reconstruct the LN7 abnormal returns, polarity and event study

In [ ]:
# Adaptation for development. The lab’s CSV provides the event types.
# Reconstruct independently of labels. The focus is on utilizing the available range/selection.
lab_path = Path("data/Data for laboratories-20261119/sentratio_and_price_st.csv")
lab = pd.read_csv(lab_path, parse_dates=["date"])
lab = lab.loc[lab.businessday & (lab.date < "2016-01-01")]
def panel(column):
    return lab.pivot(index="date", columns="ticker", values=column).sort_index()

volume, polarity, lab_returns = (panel(col) for col in ["N", "Polarity", "daily_return"])
polarity = polarity.where(volume > 0)  # Unknown polarity in the absence of messages.
total_volume = volume.sum(axis=1, min_count=1)
market_polarity = (volume * polarity).sum(axis=1, min_count=1) / volume.where(polarity.notna()).sum(axis=1, min_count=1)

def past_market_residual(series, market):
    # Each firm-market pair uses exactly the same observations for the moments.
    x = market.where(series.notna())
    y = series.where(x.notna())
    def mean(s):
        return s.rolling("365D", closed="left", min_periods=120).mean()
    variance = mean(x*x) - mean(x)**2
    beta = (mean(x*y) - mean(x)*mean(y)) / variance.where(variance > 1e-12)
    return y - mean(y) - beta * (x - mean(x))

market_growth = total_volume.pct_change(fill_method=None)
growth = volume.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
av = growth.apply(past_market_residual, market=market_growth)
ap = polarity.apply(past_market_residual, market=market_polarity)
ar = lab_returns.apply(past_market_residual, market=lab_returns["SPY"])
baseline = av.rolling("365D", closed="left", min_periods=120)
peaks = av.gt(baseline.mean() + 2 * baseline.std()) & ap.notna()
reference = panel("Type").notna()
eligible = baseline.std().gt(0) & av.notna() & ap.notna()
peaks &= eligible
print("Reconstructed peaks / supplied events / matches on eligible dates:",
      peaks.sum().sum(), (reference & eligible).sum().sum(), (reference & peaks).sum().sum())

# The labels use the polarity of day 0 and are intended solely for retrospective study.
events = []
prior_events = []
prior_cap = ap.rolling(20, min_periods=20).sum().shift(1)
for ticker in peaks.columns.drop("SPY"):
    for day in np.flatnonzero(peaks[ticker]):
        value = ap[ticker].iloc[day]
        kind = "Good" if value > .07 else "Bad" if value < -.03 else "Neutral"
        cap = prior_cap[ticker].iloc[day]
        if pd.notna(cap):
            prior_events.append((ap.index[day], ticker, kind, cap))
        if day < 20 or day + 20 >= len(ar):
            continue
        sample = pd.DataFrame({"CAR": ar[ticker], "CAP": ap[ticker]}).iloc[day-20:day+21]
        if sample.isna().any().any():
            continue
        event = sample.cumsum()
        event["offset"] = np.arange(-20, 21)
        event["ticker"], event["date"] = ticker, ar.index[day]
        event["type"] = kind
        events.append(event)
event_paths = pd.concat(events, ignore_index=True)
means = event_paths.groupby(["offset", "type"])[["CAR", "CAP"]].mean()
means.unstack().plot(subplots=True, figsize=(8, 5), title="Cumulative means by type around the peak")
# Comparison before the event. The nominal p-value ignores the dependence between events.
prior = pd.DataFrame(prior_events, columns=["date", "ticker", "type", "CAP"])
# For complete windows, the two CAP calculations on J-1 must align.
full_prior = event_paths.query("offset == -1").merge(prior, on=["date", "ticker", "type"],
                                                     validate="one_to_one", suffixes=("_path", "_prior"))
assert len(full_prior) == len(events)
np.testing.assert_allclose(full_prior.CAP_path, full_prior.CAP_prior, atol=1e-10)
print("Complete windows from day −20 to day +20:", len(events))
rank_test = mannwhitneyu(prior.loc[prior.type == "Good", "CAP"], prior.loc[prior.type == "Neutral", "CAP"])
print("Eligible events on day −1 by type:", prior.type.value_counts().to_dict())
print("Mann–Whitney Good/Neutral, CAP at day −1, nominal p-value:", round(rank_test.pvalue, 4))
print("Dates shared by several events:", int((prior.groupby("date").size() > 1).sum()))
# Descriptive sensitivity: groups re-sampled together in blocks of sessions.
# Include days without events to preserve block length.
by_type = prior.pivot_table(index="date", columns="type", values="CAP", aggfunc=["sum","count"])
by_type = by_type.reindex(ap.index).fillna(0)
def mean_difference(values):
    sums = values.sum(axis=0)
    if sums[1] == 0 or sums[3] == 0:
        return np.nan
    return sums[0]/sums[1] - sums[2]/sums[3]
values = by_type.loc[:,[("sum","Good"),("count","Good"),("sum","Neutral"),("count","Neutral")]].to_numpy()
rng = np.random.default_rng(207)
for length in (20,40):
    samples = []
    for _ in range(1500):
        starts = rng.integers(len(values), size=int(np.ceil(len(values)/length)))
        ix = ((starts[:,None]+np.arange(length))%len(values)).ravel()[:len(values)]
        samples.append(mean_difference(values[ix]))
    print("Mean CAP for Good minus Neutral at day −1, block length, exploratory CI:",
          mean_difference(values), length, np.nanquantile(samples,[.025,.975]))

# Sensitivity to one’s own weight within the polarity factor. Maintaining the same events.
# and labels to isolate the effect of the reference change on CAP at J-1.
weighted = volume*polarity
matched_volume = volume.where(polarity.notna())
total_weight = weighted.sum(axis=1,min_count=1)
total_count = matched_volume.sum(axis=1,min_count=1)
peer_cap = {}
for ticker in peaks.columns.drop("SPY"):
    peer_count = total_count - matched_volume[ticker].fillna(0)
    peer_polarity = (total_weight-weighted[ticker].fillna(0))/peer_count.where(peer_count>0)
    peer_cap[ticker] = past_market_residual(polarity[ticker],peer_polarity).rolling(20).sum().shift(1)
peer_panel = pd.DataFrame(peer_cap)
peer_panel.index.name = "date"
peer_panel.columns.name = "ticker"
paired = prior.merge(peer_panel.stack().rename("CAP_pairs").reset_index(),on=["date","ticker"],
                     validate="one_to_one")
print("Events matched with a market series excluding the stock:", len(paired))
print(paired.groupby("type")[["CAP","CAP_pairs"]].mean().round(3))


### 5.0 — Shared Functions and Quick Tests (Euler, ERC, BL Without Views)

In [ ]:
# Reproducible controls in USD, based solely on the development history.
prices = pd.read_csv("data/raw/etf_prices_adjusted.csv", index_col="date", parse_dates=True)
returns = prices.loc["2005":"2010", ["SPY", "IEF", "GLD"]].pct_change(fill_method=None).dropna()
train, evaluation = returns.loc[:"2009"], returns.loc["2010"]
mu = train.mean().to_numpy() * 252
cov = train.cov().to_numpy() * 252
held = np.full(len(mu), 1/len(mu))
# One arbitration for a one-year horizon. Fees are indicative per dollar traded.
fee = .001
def allocation(expected, covariance, phi, *, long_only=True, cost=0, target=None, turnover_cap=None):
    w = cp.Variable(len(expected))
    traded = cp.norm1(w - held)
    net = expected @ w - cost * traded
    constraints = [cp.sum(w) == 1]
    if long_only:
        constraints.append(w >= 0)
    if target is not None:
        constraints.append(net >= target)
    if turnover_cap is not None:
        constraints.append(traded <= turnover_cap)
    problem = cp.Problem(cp.Maximize(net - phi * cp.quad_form(w, covariance)/2), constraints)
    problem.solve(solver="CLARABEL")
    if problem.status == cp.INFEASIBLE:
        return None
    if problem.status != cp.OPTIMAL:
        raise RuntimeError(problem.status)
    return w.value

ones = np.ones(len(mu))
for phi in [2, 5, 10]:
    inv_mu = np.linalg.solve(cov, mu)
    inv_one = np.linalg.solve(cov, ones)
    multiplier = (ones @ inv_mu - phi) / (ones @ inv_one)
    analytic = (inv_mu - multiplier * inv_one) / phi
    np.testing.assert_allclose(allocation(mu, cov, phi, long_only=False), analytic, atol=1e-6)
    weights = allocation(mu, cov, phi, cost=fee)
    assert min(weights) >= -1e-7 and abs(weights.sum()-1) < 1e-7
    print("φ =", phi, "SPY/IEF/GLD weights", weights.round(3))
assert allocation(mu, cov, 5, cost=fee, target=max(mu)+.01) is None

### 5.2 — Equal risk contribution (ERC)

In [ ]:
def erc(matrix):
    x = cp.Variable(len(matrix), pos=True)
    problem = cp.Problem(cp.Minimize(cp.quad_form(x, matrix)/2 - cp.sum(cp.log(x))/len(matrix)))
    problem.solve(solver="CLARABEL")
    if problem.status != cp.OPTIMAL:
        raise RuntimeError(problem.status)
    return x.value / x.value.sum()

sigma = np.sqrt(np.diag(cov))
uniform_corr = .25 + .75*np.eye(len(mu))
erc_uniform = erc(np.outer(sigma, sigma) * uniform_corr)
np.testing.assert_allclose(erc_uniform, (1/sigma)/(1/sigma).sum(), atol=1e-4)
erc_weights = erc(cov)
risk_shares = erc_weights * (cov @ erc_weights) / (erc_weights @ cov @ erc_weights)
np.testing.assert_allclose(risk_shares, held, atol=1e-4)
print("ERC risk contributions:", risk_shares.round(4))

### 5.7 — Portfolio Analytics (Weights, RC/RC*, Volatility, TE, IR, Concentration)

In [ ]:
def active_metrics(weights, benchmark):
    active = weights - benchmark
    active_return = evaluation @ active
    realized = active_return.std() * np.sqrt(252)
    predicted = np.sqrt(active @ cov @ active)
    info_ratio = active_return.mean() * 252 / realized if realized > 0 else np.nan
    return predicted, realized, info_ratio

te_forecast, te_realized, ir = active_metrics(held, held)
assert te_forecast == te_realized == 0 and np.isnan(ir)
print("Forecast TE, realized TE in 2010, gross information ratio:", active_metrics(weights, held))

### 5.10 — Expected Shortfall and diversification (LN6)

In [ ]:
def diversification(weights, covariance):
    volatility = np.sqrt(weights @ covariance @ weights)
    return 1/(weights @ weights), weights @ np.sqrt(np.diag(covariance))/volatility if volatility > 0 else np.nan

twins = diversification(np.array([.5, .5]), np.full((2, 2), .04))
np.testing.assert_allclose(twins, [2, 1])
print("Effective number of holdings and diversification ratio for the ERC portfolio:",
      diversification(erc_weights, cov))

### 5.12 — Portfolio constraint and solver audit

In [ ]:
regularized = LedoitWolf().fit(train).covariance_ * 252
base = allocation(mu, cov, 5, cost=fee)
# Establish the risk ceiling by aligning it with the benchmark. Compare each disruption against the same standard.
risk_cap = np.sqrt(base @ cov @ base)
for label, expected, estimate, limit in [
    ("baseline", mu, cov, 2), ("equity μ +1 percentage point", mu + [.01, 0, 0], cov, 2),
    ("regularized Σ", mu, regularized, 2), ("capped turnover", mu, cov, .10)]:
    w = cp.Variable(len(mu))
    traded = cp.norm1(w-held)
    problem = cp.Problem(cp.Maximize(expected @ w - fee*traded - 2.5*cp.quad_form(w, estimate)),
                         [w >= 0, cp.sum(w)==1, traded <= limit,
                          cp.quad_form(w, cov) <= risk_cap**2])
    problem.solve(solver="CLARABEL")
    assert problem.status == cp.OPTIMAL
    turnover = np.abs(w.value-held).sum()
    assert turnover <= limit + 1e-6 and w.value @ cov @ w.value <= risk_cap**2 + 1e-7
    print(label, "weights", w.value.round(3), "gross turnover", round(turnover, 4),
          "cost (bp)", round(1e4 * fee * turnover, 2))

### 5.13 — Black–Litterman no-view identity and confidence response

In [ ]:
# Avoidance of the benchmark portfolio is used for prioritizing excess returns.
market_phi, tau = 3.0, .05
prior_mean = market_phi * cov @ held
view = np.array([1., -1., 0.])
q = .025
def posterior(omega):
    if omega is None:
        return prior_mean.copy()
    cov_mean = tau * cov
    return prior_mean + cov_mean @ view * ((q-view@prior_mean)/(view@cov_mean@view+omega))

np.testing.assert_array_equal(posterior(None), prior_mean)
assert view.sum() == 0
errors = [np.linalg.norm(posterior(omega)-prior_mean) for omega in [1e-6, .01, 1e6]]
assert errors[0] > errors[1] > errors[2] and errors[2] < 1e-8
print("Deviations from the prior as confidence falls:", errors)

### H03 — Sensitivity of each allocation to the same interest-rate move

In [ ]:
import statsmodels.api as sm

yields = pd.read_csv("data/raw/public_phase2/DGS10.csv", parse_dates=["observation_date"],
                     index_col="observation_date")["DGS10"]
# Both variations have the same closing range. A missing observation eliminates the pair.
change = yields.reindex(prices.index).diff().rename("rate_change")
observations = train.join(change).dropna()
design = sm.add_constant(observations["rate_change"])
asset_returns = observations[train.columns]
shock = .10  # DGS10 is expressed in percentage points.
asset_effects = np.linalg.lstsq(design, asset_returns, rcond=None)[0][1] * shock
examples = {"1/N": held, "ERC": erc_weights}
for phi in (2, 5, 10):
    examples[f"φ = {phi}"] = allocation(mu, cov, phi, cost=fee)

records = []
for name, composition in examples.items():
    model = sm.OLS(asset_returns @ composition, design).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    effect = model.params["rate_change"] * shock
    contributions = composition * asset_effects
    np.testing.assert_allclose(contributions.sum(), effect, atol=1e-10)
    lower, upper = model.conf_int().loc["rate_change"] * shock
    records.append([name, *contributions, effect, lower, upper])
scenario = pd.DataFrame(records, columns=["Composition", *train.columns, "Total", "Lower CI", "Upper CI"])
print("Estimated return impact (percentage points), asset contributions and CI for the total")
print((100 * scenario.set_index("Composition")).round(3))

### 6.1 — Monthly Backtest Engine

In [ ]:
from scipy.optimize import brentq

def minimum_variance_weights(covariance):
    w = cp.Variable(len(benchmark_assets))
    problem = cp.Problem(cp.Minimize(cp.quad_form(w, covariance / np.max(np.diag(covariance)))),
                         [cp.sum(w) == 1, w >= 0])
    problem.solve(solver="CLARABEL")
    if problem.status != cp.OPTIMAL or w.value.min() < -1e-7:
        raise RuntimeError("Invalid GMV solution")
    gmv = np.maximum(w.value, 0)
    return gmv / gmv.sum()

def benchmark_targets(daily, start, end):
    if not daily.index.is_unique or not daily.index.is_monotonic_increasing:
        raise ValueError("Invalid dates")
    verify_calendar(daily.index)
    if list(daily.columns) != benchmark_assets:
        raise ValueError("Expected order is SPY, IEF, GLD")
    targets = []
    # The first observed meeting of the month implements the previous closure decision.
    months = daily.index.to_period("M")
    for idx in np.flatnonzero(months[1:] != months[:-1]) + 1:
        execution = daily.index[idx]
        if not start <= execution <= end:
            continue
        if idx < cov_window:
            raise ValueError("Insufficient history for GMV")
        history = daily.iloc[idx-cov_window:idx]
        covariance = history.cov().to_numpy()
        gmv = minimum_variance_weights(covariance)
        allocations = {"60/40": [.6, .4, 0], "1/N": [1/3]*3, "GMV": gmv}
        for name, target in allocations.items():
            targets.append([name, history.index[-1], execution, *target])
    return pd.DataFrame(targets, columns=["strategy", "decision", "execution", *benchmark_assets])

def monthly_account(daily, orders, fee_bps):
    # The positions are represented by amounts in USD. The initial cash of $100 is invested at the first execution.
    # Resolving costs before calculating revenue allows you to accurately bill for the exact amounts that were actually negotiated.
    fee_rate = np.broadcast_to(np.asarray(fee_bps, dtype=float)/10_000, (daily.shape[1],))
    if not np.all((fee_rate >= 0) & (fee_rate < 1)) or daily.empty or orders.empty:
        raise ValueError("Invalid fee schedule or calendar")
    orders = orders.set_index("execution", verify_integrity=True)
    if (not orders.index.isin(daily.index).all() or orders.index.min() != daily.index[0]
            or not (orders["decision"] < orders.index).all()):
        raise ValueError("Orders do not match the pilot trading sessions")
    if not np.isfinite(daily.to_numpy()).all() or (daily.to_numpy() < -1).any():
        raise ValueError("Invalid returns")
    positions = np.zeros(daily.shape[1])
    cash, previous = 100., 100.
    ledger, trades = [], []
    for day, day_returns in zip(daily.index, daily.to_numpy()):
        positions *= 1 + day_returns
        gross_value = cash + positions.sum()
        cost = 0.
        if day in orders.index:
            order = orders.loc[day]
            target = order[daily.columns].to_numpy(dtype=float)
            if (target < 0).any() or not np.isclose(target.sum(), 1):
                raise ValueError("Invalid target weights")
            def balance(after):
                return after + fee_rate @ np.abs(after * target - positions) - gross_value
            after = gross_value if np.all(fee_rate == 0) else brentq(balance, 0, gross_value)
            delta = after * target - positions
            cost = fee_rate @ np.abs(delta)
            if abs(cash - delta.sum() - cost) > 1e-8:
                raise ArithmeticError("Transaction is not self-financing")
            trades.append([order["decision"], day, gross_value, cost, *delta])
            positions, cash = after * target, 0.
        closing = positions.sum() + cash
        if abs(closing + cost - gross_value) > 1e-8:
            raise ArithmeticError("Wealth reconciliation failed")
        ledger.append([day, closing, closing/previous-1, cost])
        previous = closing
    wallet = pd.DataFrame(ledger, columns=["date", "capital", "asset_return", "fees"]).set_index("date")
    transactions = pd.DataFrame(trades, columns=["decision", "execution", "capital_before_fees", "fees", *daily.columns])
    return wallet, transactions

### 6.3 — Transaction Costs

In [ ]:
daily_pilot = prices.loc[:pilot_end, benchmark_assets].pct_change(fill_method=None).iloc[1:]
orders = benchmark_targets(daily_pilot, pilot_start, pilot_end)
pilot_returns = daily_pilot.loc[pilot_start:pilot_end]
assert pilot_returns.index.max() < prospective_start
records = []
pilot_trades = {}
for strategy, schedule in orders.groupby("strategy"):
    assert len(schedule) == len(pd.period_range(pilot_start, pilot_end, freq="M"))
    for fee_bps in benchmark["fee_bps"]:
        wallet, transactions = monthly_account(pilot_returns, schedule, fee_bps)
        pilot_trades[strategy, fee_bps] = transactions
        capital = wallet.capital.iloc[-1]
        np.testing.assert_allclose((1 + wallet.asset_return).prod(), capital / 100)
        peak = wallet.capital.cummax().clip(lower=100)
        records.append([strategy, fee_bps, len(schedule), capital, wallet.fees.sum(),
                        wallet.asset_return.std()*np.sqrt(252), (wallet.capital/peak-1).min()])
summary = pd.DataFrame(records, columns=["strategy", "fees_bp", "orders", "final_capital",
                                         "cumulative_fees_USD", "annual_vol", "drawdown"])
for _, variants in summary.groupby("strategy"):
    assert variants.final_capital.is_monotonic_decreasing
print(summary.round(4).to_string(index=False))

### Guides 01, 04 and 05 — Did the stated loss threshold hold?

In [ ]:
def empirical_tail(losses, tail_mass):
    """Empirical quantile and tail mean, including the relevant fraction of the boundary observation."""
    descending = np.sort(losses)[::-1]
    mass = len(descending) * tail_mass
    if not 0 < mass <= len(descending) or not np.isfinite(descending).all():
        raise ValueError("Invalid tail or observations")
    whole = int(np.floor(mass))
    fraction = mass - whole
    # Left quantile of the cumulative distribution, including when the mass is integer.
    threshold = descending[min(whole, len(descending)-1)]
    total = descending[:whole].sum()
    if fraction:
        total += fraction * threshold
    return threshold, total / mass

np.testing.assert_allclose(empirical_tail([0, 1, 1, 4], .375), [1, 3])
np.testing.assert_allclose(empirical_tail([0, 1, 1, 4], .25), [1, 4])
tail_records = []
for name, composition in examples.items():
    losses = -(returns @ composition)
    for date in evaluation.index:
        index = losses.index.get_loc(date)
        history = losses.iloc[index-252:index]
        assert len(history) == 252 and history.index[-1] < date
        var, es = empirical_tail(history, .05)
        assert es >= var - 1e-12
        tail_records.append((date, name, var, es, losses.loc[date]))

risk_ledger = pd.DataFrame(tail_records, columns=["date", "composition", "VaR", "ES", "loss"])
def tail_summary(group):
    hits = group.loss > group.VaR
    runs = hits.groupby((~hits).cumsum()).sum()
    excess = (group.loss - group.VaR)[hits]
    return pd.Series({"sessions": len(group), "exceedances": hits.sum(),
                      "frequency (%)": 100 * hits.mean(), "max run": runs.max(),
                      "mean excess given exceedance (bp)": 10000 * excess.mean()})

print(risk_ledger.groupby("composition")[["VaR", "ES", "loss"]].apply(tail_summary).round(2))
print("ES recorded by date. Exceedance frequency evaluates VaR.")

### 6.5 — Robustness

In [ ]:
# Modifying all returns upon execution leaves previously made decisions unchanged.
cut = orders["execution"].sort_values().unique()[12]
changed = daily_pilot.copy()
changed.loc[cut:, "SPY"] += .10
before = orders.loc[orders["execution"] <= cut].reset_index(drop=True)
pd.testing.assert_frame_equal(before, benchmark_targets(changed, pilot_start, cut))

# The initial purchase doesn't guarantee a rise on the execution day. In an arbitration…
# The old positions still capture today’s fluctuations, and the new ones capture those of tomorrow.
days = pd.bdate_range("2015-02-02", periods=4)
toy = pd.DataFrame([[1., 0.], [.10, 0.], [-.50, .50], [0., .20]],
                   index=days, columns=["A", "B"])
toy_orders = pd.DataFrame([[days[0]-pd.Timedelta(days=3),days[0],1.,0.],
                           [days[1],days[2],0.,1.]], columns=["decision", "execution", "A", "B"])
path, _ = monthly_account(toy, toy_orders, 0)
np.testing.assert_allclose(path.capital, [100, 110, 55, 66])
# Fixed prices: charge the entry fee based on the amount purchased; re-sending the same target doesn't cost anything.
flat_orders = toy_orders.copy()
flat_orders[["A", "B"]] = [1., 0.]
path, trades = monthly_account(toy*0, flat_orders, 10)
np.testing.assert_allclose(path.capital, 100/1.001)
np.testing.assert_allclose(trades.fees.iloc[-1], 0, atol=1e-10)
print("Chronology, self-financing and invariance to future data checked.")
try:
    benchmark_targets(daily_pilot.drop(index=orders.execution.iloc[3]), pilot_start, pilot_end)
except ValueError as exc:
    assert "XNYS" in str(exc)
else:
    raise AssertionError("A missing session was accepted")


### H01 — Covariance estimation and realized performance

In [ ]:
def circular_blocks(length, block, draws, rng):
    if not 1 <= block <= length:
        raise ValueError("Invalid block length")
    starts = rng.integers(length, size=(draws, int(np.ceil(length/block)), 1))
    return ((starts + np.arange(block)) % length).reshape(draws, -1)[:, :length]

equal = np.full(len(benchmark_assets), 1/len(benchmark_assets))
sample_orders = orders.loc[orders.strategy == "GMV"].copy()
lw_orders = sample_orders.copy()
risk_forecasts = []
for i, order in sample_orders.iterrows():
    history = daily_pilot.loc[:order.decision].tail(cov_window)
    empirical = history.cov().to_numpy()
    regularized = LedoitWolf().fit(history).covariance_ * cov_window/(cov_window-1)
    lw_orders.loc[i, benchmark_assets] = minimum_variance_weights(regularized)
    future = pilot_returns.loc[str(order.execution.to_period("M"))]
    realized = ((future @ equal)**2).sum()
    row = {"decision": order.decision, "realized_variance": realized}
    for label, matrix in [("historical", empirical), ("LW", regularized)]:
        predicted = len(future) * equal @ matrix @ equal
        row["QLIKE_" + label] = np.log(predicted) + realized/predicted
        row["condition_" + label] = np.linalg.cond(matrix)
    risk_forecasts.append(row)
risk_forecasts = pd.DataFrame(risk_forecasts).set_index("decision")
gain = (risk_forecasts.QLIKE_historical - risk_forecasts.QLIKE_LW).to_numpy()
print("Mean QLIKE gain in favor of LW:", gain.mean())
rng_h01 = np.random.default_rng(setup["seeds"]["covariance_bootstrap"])
for block in (3, 6, 12):
    bootstrap = gain[circular_blocks(len(gain), block, 5000, rng_h01)].mean(axis=1)
    print("Block length in months and 95% interval:", block, np.quantile(bootstrap, [.025, .975]))
for label, schedule in [("historical", sample_orders), ("LW", lw_orders)]:
    path, trades = monthly_account(pilot_returns, schedule, 10)
    print(label, "Median condition number, final capital, net volatility, fees:",
          risk_forecasts["condition_"+label].median(), path.capital.iloc[-1],
          path.asset_return.std()*np.sqrt(252), trades.fees.sum())

### H07 — Net optimum and minimum trading

In [ ]:
class AllocationReview:
    """Two reusable convex programs with the same universe and risk aversion."""
    def __init__(self, n, phi, cost=.001):
        self.phi, self.cost = phi, cost
        self.w = cp.Variable(n)
        self.mean = cp.Parameter(n)
        self.factor = cp.Parameter((n, n))
        self.held = cp.Parameter(n)
        self.cap = cp.Parameter(nonneg=True)
        self.floor = cp.Parameter()
        traded_assets = np.ones(n) if np.isscalar(cost) else (np.asarray(cost)>0).astype(float)
        turnover = cp.norm1(cp.multiply(traded_assets, self.w-self.held))
        risk = cp.norm(self.factor @ self.w, 2)
        estimated_cost = cp.sum(cp.multiply(cost,cp.abs(self.w-self.held)))
        utility = self.mean @ self.w - phi*cp.sum_squares(self.factor @ self.w)/2 - estimated_cost
        constraints = [self.w >= 0, cp.sum(self.w) == 1, risk <= self.cap]
        self.best = cp.Problem(cp.Maximize(utility), constraints)
        self.near = cp.Problem(cp.Minimize(turnover), [*constraints, utility >= self.floor])

    def evaluate(self, w):
        risk = np.linalg.norm(self.factor.value @ w)
        utility = self.mean.value @ w - self.phi*risk**2/2 - np.sum(self.cost*np.abs(w-self.held.value))
        return utility, risk

    def solve(self, problem):
        problem.solve(solver="CLARABEL")
        if problem.status == cp.INFEASIBLE:
            raise ValueError("Mandate infeasible under this covariance matrix")
        if problem.status != cp.OPTIMAL:
            raise RuntimeError(problem.status)
        w = self.w.value.copy()
        if w.min() < -1e-7 or abs(w.sum()-1) > 1e-7:
            raise ArithmeticError("Solution outside mandate constraints")
        w = np.maximum(w, 0)
        return w/w.sum()

    def optimum(self, history, held, cap=None):
        self.mean.value = history.mean(axis=0)*252
        cov = np.cov(history, rowvar=False)*252
        eigenvalues, vectors = np.linalg.eigh(cov)
        if eigenvalues.min() < -1e-10:
            raise ValueError("Covariance matrix is not positive semidefinite")
        self.factor.value = np.sqrt(eigenvalues.clip(0))[:, None] * vectors.T
        self.held.value = held
        # In a long-only strategy, no combination exceeds the volatility of any individual asset.
        self.cap.value = np.sqrt(np.diag(cov).max()) if cap is None else cap
        target = self.solve(self.best)
        if self.evaluate(target)[1] > self.cap.value + 1e-7:
            raise ArithmeticError("Risk cap exceeded")
        return target

    def smallest_trade(self, optimum, tolerance, mandate_cap=None):
        if tolerance < 0:
            raise ValueError("Negative tolerance")
        best, risk = self.evaluate(optimum)
        self.floor.value = best - tolerance
        risk = risk if mandate_cap is None else mandate_cap
        self.cap.value = risk
        current = self.held.value
        current_score, current_risk = self.evaluate(current)
        if current_score >= self.floor.value-1e-10 and current_risk <= risk+1e-10:
            return current.copy()
        if tolerance == 0:
            return optimum.copy()
        target = self.solve(self.near)
        score, candidate_risk = self.evaluate(target)
        if score < self.floor.value-1e-7 or candidate_risk > risk+1e-7:
            raise ArithmeticError("Near-optimal solution exceeds tolerance or risk cap")
        return target

### Allocation dispersion at the December 31, 2015 decision

In [ ]:
sensitivity_history = daily_pilot.tail(cov_window).to_numpy()
rng_weights = np.random.default_rng(setup["seeds"]["covariance_bootstrap"])
draws_by_length = {k: circular_blocks(cov_window, k, 500, rng_weights) for k in (10, 20, 40)}
sensitivity_rows = []
for phi in (2, 5, 10):
    review = AllocationReview(len(equal), phi)
    point = review.optimum(sensitivity_history, equal)
    for block, indices in draws_by_length.items():
        simulated, regrets = [], []
        for idx in indices:
            target = review.optimum(sensitivity_history[idx], equal)
            regret = review.evaluate(target)[0] - review.evaluate(point)[0]
            assert regret >= -1e-7
            regrets.append(10000*regret)
            simulated.append(target)
        lo, hi = np.quantile(simulated, [.1, .9], axis=0)
        for j, asset in enumerate(benchmark_assets):
            sensitivity_rows.append([phi, block, asset, point[j], lo[j], hi[j], np.quantile(regrets,.9)])
sensitivity = pd.DataFrame(sensitivity_rows, columns=["phi","block","ETF","weights","p10","p90","regret_p90_bp"])
print(sensitivity.query("block == 20").round(3).to_string(index=False))

review = AllocationReview(len(equal), 5)
point = review.optimum(sensitivity_history, equal)
for tolerance in (.001, .0025, .005):
    near = review.smallest_trade(point, tolerance)
    print("Tolerance (bp), near-optimal target, purchases + sales:", tolerance*10000, near.round(3),
          np.abs(near-equal).sum().round(3))
np.testing.assert_allclose(review.smallest_trade(point, 0), point)
# Two assets with the same mean, variance, and covariance: the solution is 50/50.
toy_history = np.array([[.01, 0], [-.01, 0], [0, .01], [0, -.01]])
two = AllocationReview(2, 5, cost=0)
balanced = np.array([.5, .5])
toy_target = two.optimum(toy_history, balanced)
np.testing.assert_allclose(toy_target, balanced, atol=1e-6)
np.testing.assert_array_equal(two.smallest_trade(toy_target, .0025), balanced)
try:
    two.optimum(toy_history, balanced, cap=.001)
except ValueError as exc:
    assert "infeasible" in str(exc)
else:
    raise AssertionError("An infeasible cap was accepted")

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, sharey=True, figsize=(9, 3))
for ax, (phi, group) in zip(axes, sensitivity.query("block == 20").groupby("phi")):
    ax.hlines(group.ETF, group.p10*100, group.p90*100, color="steelblue", lw=4, label="P10–P90")
    ax.scatter(group.weights*100, group.ETF, color="black", s=22, label="Initial target", zorder=3)
    ax.set(title=f"φ = {phi}", xlabel="Weights (%)", xlim=(-4, 104))
axes[0].legend(loc="center", fontsize=8)
fig.suptitle("Sensitivity of portfolio weights to historical data · December 31, 2015")
fig.tight_layout()
plt.show()

# Three hypotheses set within the pilot. Compare the average hypothesis and maintain it.
# Two allocations were actually received, as ordered based on their weighting.
origin_results = []
rng_origins = np.random.default_rng(setup["seeds"]["frontier_origins"])
for year in (2011,2013,2015):
    sample = daily_pilot.loc[:str(year)].tail(cov_window).to_numpy()
    model = AllocationReview(3,5)
    point = model.optimum(sample,equal)
    avg = sample.mean(axis=0)
    pooled = model.optimum(sample - .5*(avg-avg.mean()),equal)
    gmv = minimum_variance_weights(np.cov(sample,rowvar=False))
    trials = np.stack([model.optimum(sample[ix],equal)
                       for ix in circular_blocks(len(sample),20,200,rng_origins)])
    ordering = np.argsort(trials[:,0])
    for name, w in [("historical means",point),("means with 50% shrinkage",pooled),
                    ("GMV",gmv),("low SPY draw",trials[ordering[19]]),("high SPY draw",trials[ordering[179]])]:
        assert np.isclose(w.sum(),1) and w.min()>=0
        origin_results.append([year,name,*w])
print(pd.DataFrame(origin_results,columns=["end","variant",*benchmark_assets]).round(3).to_string(index=False))


### Monthly decision comparison, 2010–2015

In [ ]:
def review_schedule(daily, calendar, policy, model, risk_cap, tolerance):
    if policy not in {"hold", "optimum", "minimal", "minimal mandate"}:
        raise ValueError(policy)
    filled, journal = [], []
    last_fill = None
    last_weights = equal.copy()
    for _, order in calendar.iterrows():
        decision, execution = order.decision, order.execution
        execute = True
        loss_bp = np.nan
        if last_fill is None:
            held_now = np.zeros(len(equal))
            history = daily.loc[:decision].tail(cov_window).to_numpy()
            target = model.optimum(history, held_now, risk_cap)
        else:
            since_fill = daily.loc[(daily.index > last_fill) & (daily.index <= decision)]
            values = last_weights * (1+since_fill).prod().to_numpy()
            held_now = values/values.sum()
            history = daily.loc[:decision].tail(cov_window).to_numpy()
            if policy == "hold":
                target = held_now
            else:
                point = model.optimum(history, held_now, risk_cap)
                target = point
                if policy.startswith("minimal"):
                    cap = risk_cap if policy == "minimal mandate" else None
                    target = model.smallest_trade(point,tolerance,mandate_cap=cap)
                loss_bp = 10_000 * (model.evaluate(point)[0] - model.evaluate(target)[0])
            execute = np.abs(target-held_now).sum() > 1e-8
        predicted_risk = np.std(history @ target, ddof=1)*np.sqrt(252)
        journal.append([decision, execution, execute, predicted_risk, loss_bp,
                        np.abs(target-held_now).sum(), *held_now, *target])
        if not execute:
            continue
        filled.append([decision, execution, *target])
        last_weights, last_fill = target, execution
    journal_columns = ["decision","next_session","order","estimated_risk","utility_gap_bp",
                       "estimated_turnover", *["before_"+a for a in benchmark_assets],
                       *["target_"+a for a in benchmark_assets]]
    return (pd.DataFrame(filled, columns=["decision","execution",*benchmark_assets]),
            pd.DataFrame(journal, columns=journal_columns))

risk_cap, tolerance = .12, .0025
policy_wallets, policy_orders, policy_reviews, policy_metrics = {}, {}, {}, []
for policy in ("hold", "optimum", "minimal", "minimal mandate"):
    schedule, journal = review_schedule(daily_pilot, sample_orders, policy, AllocationReview(3, 5),
                               risk_cap, tolerance)
    policy_reviews[policy] = journal
    assert len(journal) == len(sample_orders) and journal.order.sum() == len(schedule)
    if policy.startswith("minimal"):
        assert journal.utility_gap_bp.max() <= tolerance*10_000 + .001
    wallet, trades = monthly_account(pilot_returns, schedule, 10)
    policy_wallets[policy], policy_orders[policy] = wallet, schedule
    monthly_risk = wallet.asset_return.groupby(wallet.index.to_period("M")).std()*np.sqrt(252)
    turnover = trades[benchmark_assets].abs().sum(axis=1)/trades.capital_before_fees
    policy_metrics.append([policy, len(trades)-1, turnover.sum(), trades.fees.sum(),
                           wallet.capital.iloc[-1], int((monthly_risk > risk_cap).sum())])
policy_table = pd.DataFrame(policy_metrics, columns=["rule","rebalances_after_entry","cumulative_turnover",
                                                     "fees_USD","final_capital_USD","months_above_12pct"])
print(policy_table.round(3).to_string(index=False))

# Any subsequent modification to the decision cannot alter the established order.
cutoff = sample_orders.execution.iloc[12]
prefix = sample_orders.loc[sample_orders.execution <= cutoff]
altered = daily_pilot.copy()
altered.loc[cutoff:, "SPY"] += .10
replayed, prefix_journal = review_schedule(altered, prefix, "minimal", AllocationReview(3, 5), risk_cap, tolerance)
pd.testing.assert_frame_equal(policy_orders["minimal"].query("execution <= @cutoff").reset_index(drop=True), replayed)
assert len(policy_orders["hold"]) == 1
pd.testing.assert_frame_equal(policy_reviews["minimal"].iloc[:len(prefix)], prefix_journal)

assert all(j.estimated_risk.iloc[0] <= risk_cap+1e-7 for j in policy_reviews.values())


### Guides 05, 26, 33 and 34 — Reserves, withdrawal and the sequence of losses

In [ ]:
def retirement_path(reserve, before, after, sale_allowed=True):
    initial_capital, due = 50_000, 15_000
    risky = (initial_capital - reserve) * (1 + before)
    pre_withdrawal = risky + reserve
    cash_used = min(reserve, due)
    sold = min(risky, due - cash_used) if sale_allowed else 0
    paid = cash_used + sold
    cash_left, risky_left = reserve - cash_used, risky - sold
    assert np.isclose(pre_withdrawal, cash_left + risky_left + paid)
    return paid, due - paid, sold, cash_left + risky_left * (1 + after)

for reserve in (0, 15_000):
    for before, after in ((-.2, .2), (.2, -.2)):
        print("Reserve, returns, (amount paid, shortfall, sale, final wealth):",
              reserve, (before, after), retirement_path(reserve, before, after))
assert retirement_path(15_000, -.2, .2) == (15_000, 0, 0, 33_600)
assert retirement_path(0, -.2, .2)[3] == 30_000
assert retirement_path(0, .2, -.2)[3] == 36_000
# Conditional stress: No sales that can be adjusted on the date of withdrawal.
assert retirement_path(0, -.2, .2, False)[1] == 15_000
assert retirement_path(15_000, -.2, .2, False)[1] == 0

### CHF mandate — Two-year withdrawal on the pilot’s historical path

In [ ]:
fx = pd.read_csv("data/raw/public_phase2/DEXSZUS.csv", index_col="observation_date",
                 parse_dates=True)["DEXSZUS"]
entry = pilot_returns.index[0]
withdrawal = entry + pd.DateOffset(years=2)
if withdrawal not in pilot_returns.index or fx.reindex([entry, withdrawal]).isna().any():
    raise ValueError("Valuation or exchange-rate date unavailable")
reserve_chf, invested_chf = 15_000., 35_000.
fx_charge_chf = invested_chf * .001
initial_usd = (invested_chf - fx_charge_chf) / fx.at[entry]
np.testing.assert_allclose(initial_usd * fx.at[entry] + fx_charge_chf + reserve_chf, 50_000)
for name, schedule in orders.groupby("strategy"):
    before_due = pilot_returns.loc[:withdrawal]
    wallet, trades = monthly_account(before_due, schedule.loc[schedule.execution <= withdrawal], 10)
    risky_chf = wallet.capital.iloc[-1] / 100 * initial_usd * fx.at[withdrawal]
    capital_before = reserve_chf + risky_chf
    paid_chf = 15_000.
    capital_after = risky_chf + (reserve_chf - paid_chf)
    np.testing.assert_allclose(capital_after + paid_chf, capital_before)
    print(name, withdrawal.date(), "withdrawal CHF", paid_chf,
          "sale for CHF withdrawal", 0, "remaining capital CHF", round(capital_after, 2))


### CHF mandate — Risk of the investable sleeve and the whole project

In [ ]:
# Retro pricing for accounting. Days without FX trading are indicated.
fx_daily = fx.reindex(daily_pilot.index)
observed_fx = fx_daily.notna()
fx_mark = fx_daily.ffill(limit=3)
client_prices = prices.loc[daily_pilot.index, benchmark_assets].mul(fx_mark, axis=0)
client_prices["Cash CHF"] = 1.
client_returns = client_prices.pct_change(fill_method=None)
daily_pairs = observed_fx & observed_fx.shift(1, fill_value=False)
risk_history = client_returns.where(daily_pairs)
instruments = client_prices.columns
# Fees proportional to ETFs. The model assumes that all transactions go through CHF.
fee_vector = np.array([.002, .002, .002, 0.])  # 10 titles + 10 changes, cash with no fees
lag = pd.Timedelta(days=10)  # Assumption of availability, to be replaced with dated publications.
quote_series = fx.dropna()
initial_cash = np.array([0., 0., 0., 1.])

def chf_schedule(policy):
    model = AllocationReview(4, 5, cost=fee_vector)
    last_units = None
    rows, log = [], []
    for order in sample_orders.itertuples():
        decision = order.decision
        # If the execution change is missing, maintain the order and continue until pricing is available.
        available = fx_daily.loc[order.execution:].dropna()
        if available.empty:
            raise ValueError("No execution with an available price")
        fill = available.index[0]
        cutoff = decision - lag
        window = risk_history.loc[:cutoff].tail(cov_window).dropna()
        if len(window) < .9*cov_window:
            raise ValueError("Insufficient coverage of the CHF window")
        history = window.to_numpy()
        fx_date = quote_series.loc[:cutoff].index[-1]
        estimate_prices = np.r_[prices.loc[decision, benchmark_assets]*quote_series[fx_date], 1.]
        held_now = initial_cash if last_units is None else last_units*estimate_prices/(last_units@estimate_prices)
        if last_units is None or policy != "hold":
            point = model.optimum(history, held_now, risk_cap)
            target = point
            if last_units is not None and policy.startswith("minimal"):
                target = model.smallest_trade(point, tolerance, mandate_cap=risk_cap if policy=="minimal mandate" else None)
            planned = model.evaluate(target)[1]
            assert planned <= risk_cap+1e-7
        else:
            target = held_now
            planned = np.std(history@target, ddof=1)*np.sqrt(252)
        trade = last_units is None or np.abs(target-held_now).sum()>1e-8
        log.append([decision, fill, fx_date, len(window), planned, trade, *target])
        if trade:
            rows.append([decision, fill, *target])
            last_units = target/client_prices.loc[fill].to_numpy()
    return (pd.DataFrame(rows,columns=["decision","execution",*instruments]),
            pd.DataFrame(log,columns=["decision","execution","assumed_known_FX_date",
                                      "observations","forecast_vol","order",*instruments]))

chf_results, chf_risk_ledgers, chf_wallets, chf_orders = [], {}, {}, {}
for rule in ("hold","optimum","minimal optimum","minimal mandate"):
    schedule, journal = chf_schedule(rule)
    chf_window = client_returns.loc[pilot_returns.index]
    assert schedule.execution.iloc[0] == chf_window.index[0] and np.isfinite(chf_window).all().all()
    wallet, transactions = monthly_account(chf_window, schedule, fee_vector*10000)
    chf_wallets[rule], chf_orders[rule] = wallet, schedule
    prior_capital = wallet.capital.shift(1,fill_value=100.)
    gross = (wallet.capital+wallet.fees)/prior_capital-1
    # Measuring the risk associated with returns on assets held after execution.
    # The final episode is cut off at the end of the pilot.
    log = journal.copy()
    for i,row in journal.iterrows():
        next_date = journal.iloc[i+1].execution if i+1 < len(journal) else gross.index[-1]
        period = gross.loc[(gross.index>row.execution)&(gross.index<=next_date)]
        observed = period[daily_pairs.reindex(period.index)]
        log.loc[i, ["sessions","valid_pairs","realized_vol"]] = [len(period), len(observed), observed.std()*np.sqrt(252)]
    log["excess"] = (log.realized_vol-risk_cap).clip(lower=0)
    complete = log.iloc[:-1].copy()
    assert len(complete)==71 and complete.realized_vol.notna().all()
    hits = complete.excess.gt(0)
    streak = hits.groupby((~hits).cumsum()).sum().max()
    # Entire project: to treat the reserve as a loss, even if it doesn’t come from an external source.
    risky_value = wallet.capital/100*invested_chf
    reserve_before = pd.Series(np.where(wallet.index <= withdrawal,reserve_chf,0.),index=wallet.index)
    reserve_after = reserve_before.where(wallet.index < withdrawal,0.)
    project_returns = (risky_value+reserve_before)/(risky_value.shift(1,fill_value=invested_chf)+reserve_before)-1
    np.testing.assert_allclose((risky_value+reserve_before)-(risky_value+reserve_after),
                               np.where(wallet.index==withdrawal,reserve_chf,0.))
    chf_risk_ledgers[rule] = log
    chf_results.append([rule,len(transactions)-1,wallet.capital.iloc[-1],transactions.fees.sum(),
                        int(hits.sum()),complete.loc[hits,"excess"].mean()*100,streak,
                        wallet.asset_return[daily_pairs.loc[wallet.index]].std()*np.sqrt(252)*100,
                        project_returns[daily_pairs.loc[wallet.index]].std()*np.sqrt(252)*100])
chf_table = pd.DataFrame(chf_results,columns=["rule","rebalances","value_per_100_CHF","fees_CHF","exceedances_in_71",
                                              "conditional_excess_points","max_run","sleeve_vol_pct","project_vol_pct"])
print(chf_table.round(3).to_string(index=False))
print("Missing FX pairs in the pilot:", int((~daily_pairs.loc[pilot_returns.index]).sum()))
for rule,wallet in chf_wallets.items():
    # Actual illustrative objective: hypothetical inflation of 2% per year for two years.
    for reserve, due, date, sales_allowed in [
        (15000,15000,withdrawal,True),(5000,15000,withdrawal,True),
        (5000,15000,pd.Timestamp("2011-01-04"),False),
        (5000,35000,pd.Timestamp("2011-01-04"),False),
        (15000,15000*1.02**2,withdrawal,True)]:
        risky = (50_000-reserve)*wallet.capital.at[date]/100
        last = chf_orders[rule].loc[chf_orders[rule].execution <= date].iloc[-1]
        marked = last[instruments].to_numpy(float)*(client_prices.loc[date]/client_prices.loc[last.execution]).to_numpy()
        cash_in_sleeve = risky*marked[-1]/marked.sum()
        needed = max(0,due-reserve-cash_in_sleeve)
        sale = min(risky-cash_in_sleeve, needed/(1-.002)) if sales_allowed else 0.
        paid = min(reserve+cash_in_sleeve+sale*(1-.002),due)
        remaining = reserve+risky-paid-sale*.002
        np.testing.assert_allclose(remaining+paid+sale*.002,reserve+risky)
        print(rule,date.date(), "required CHF", round(due,2), "reserve, ETF sale, shortfall, remaining wealth:",
              reserve,round(sale,2),round(due-paid,2),round(remaining,2))

# Sensitivity to follow-up instructions remains frozen. On an additional day (J+1), the targets remain.
# those of the original rule. Before the delayed entry, the capital remains in CHF.
for fee_bps in benchmark["fee_bps"]:
    for delay in (0,1):
        altered = chf_orders["minimal mandate"].copy()
        if delay:
            sessions = chf_window.index
            for idx, fill in altered.execution.items():
                later = fx_daily.loc[sessions[sessions>fill]].dropna()
                if later.empty:
                    raise ValueError("Window ends before execution")
                altered.at[idx,"execution"] = later.index[0]
        rates = np.array([fee_bps+10]*3+[0])  # Variable title fees + 10% currency exchange rate difference
        path, trades = monthly_account(chf_window.loc[altered.execution.iloc[0]:], altered, rates)
        print("Security costs (bp), delay (quoted sessions), net value per CHF 100:",
              fee_bps, delay, round(path.capital.iloc[-1],3))
